# PART 2: Final Holdout Evaluation

## ⚠️ WARNING: RUN THIS NOTEBOOK ONLY ONCE!

This is the FINAL evaluation on the holdout set. Do NOT run multiple times or peek at results during development.

**Input**: 
- Best model from Stage 3
- Optimal threshold from threshold tuning
- Holdout data (20%, UNTOUCHED until now)

**Output**: Final performance metrics

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import joblib
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Load Best Model and Optimal Threshold

In [ ]:
# Load best model
best_model = joblib.load('../models/final_best_model.pkl')
print(f"Loaded model: {type(best_model).__name__}")

# Load optimal threshold
with open('../results/optimal_threshold.json', 'r') as f:
    threshold_info = json.load(f)

optimal_threshold = threshold_info['optimal_threshold']
print(f"\nOptimal threshold: {optimal_threshold:.2f}")

# Load model performance from Stage 3
with open('../results/best_model_info.json', 'r') as f:
    cv_performance = json.load(f)

print(f"\nCross-Validation Performance (Stage 3):")
print(f"  F1-Score: {cv_performance['f1_score']:.4f}")
print(f"  ROC-AUC: {cv_performance['roc_auc']:.4f}")
print(f"  Precision: {cv_performance['precision']:.4f}")
print(f"  Recall: {cv_performance['recall']:.4f}")

## Preprocess Holdout Set

In [ ]:
# Load raw holdout data
holdout_df_raw = pd.read_csv('../data/holdout_data.csv')
print(f"Raw holdout data shape: {holdout_df_raw.shape}")

# Load preprocessing pipeline from Stage 1
try:
    preprocessing_pipeline = joblib.load('../models/best_preprocessing_pipeline.pkl')
    print("Loaded preprocessing pipeline from Stage 1")
    
    # Apply preprocessing
    X_holdout_raw = holdout_df_raw.drop('Revenue', axis=1)
    y_holdout = holdout_df_raw['Revenue'].astype(int)
    
    # Transform using the pipeline (excluding the classifier)
    X_holdout = preprocessing_pipeline.named_steps['preprocessor'].transform(
        preprocessing_pipeline.named_steps['feature_creator'].transform(X_holdout_raw)
    )
    
    # Convert to DataFrame
    feature_names = preprocessing_pipeline.named_steps['preprocessor'].get_feature_names_out()
    X_holdout = pd.DataFrame(X_holdout, columns=feature_names)
    
except:
    # Fallback: load the best features CSV format
    print("Using preprocessed holdout data...")
    # You'll need to preprocess holdout data using the same steps as train
    # For now, load a preprocessed version if available
    raise NotImplementedError("Please preprocess holdout data first using scripts/preprocess.py")

print(f"\nPreprocessed holdout shape: {X_holdout.shape}")
print(f"Target distribution:")
print(y_holdout.value_counts())

## Generate Predictions on Holdout Set

In [ ]:
# Get probability predictions
y_holdout_proba = best_model.predict_proba(X_holdout)[:, 1]

# Apply optimal threshold
y_holdout_pred = (y_holdout_proba >= optimal_threshold).astype(int)

print(f"Generated predictions for {len(y_holdout)} samples")
print(f"\nPredictions distribution:")
print(pd.Series(y_holdout_pred).value_counts())

## Compute Final Metrics

In [ ]:
# Calculate metrics
final_f1 = f1_score(y_holdout, y_holdout_pred)
final_precision = precision_score(y_holdout, y_holdout_pred)
final_recall = recall_score(y_holdout, y_holdout_pred)
final_accuracy = accuracy_score(y_holdout, y_holdout_pred)
final_roc_auc = roc_auc_score(y_holdout, y_holdout_proba)

print("\n" + "="*60)
print("FINAL HOLDOUT SET PERFORMANCE")
print("="*60)
print(f"\n{'Metric':<20} {'Value':<15}")
print("-" * 35)
print(f"{'F1-Score':<20} {final_f1:<15.4f}")
print(f"{'ROC-AUC':<20} {final_roc_auc:<15.4f}")
print(f"{'Precision':<20} {final_precision:<15.4f}")
print(f"{'Recall':<20} {final_recall:<15.4f}")
print(f"{'Accuracy':<20} {final_accuracy:<15.4f}")

# Compare with CV performance
print("\n" + "="*60)
print("COMPARISON: Cross-Validation vs. Holdout")
print("="*60)
print(f"\n{'Metric':<20} {'CV (Stage 3)':<15} {'Holdout':<15} {'Difference':<15}")
print("-" * 65)
print(f"{'F1-Score':<20} {cv_performance['f1_score']:<15.4f} {final_f1:<15.4f} {final_f1 - cv_performance['f1_score']:>+14.4f}")
print(f"{'ROC-AUC':<20} {cv_performance['roc_auc']:<15.4f} {final_roc_auc:<15.4f} {final_roc_auc - cv_performance['roc_auc']:>+14.4f}")
print(f"{'Precision':<20} {cv_performance['precision']:<15.4f} {final_precision:<15.4f} {final_precision - cv_performance['precision']:>+14.4f}")
print(f"{'Recall':<20} {cv_performance['recall']:<15.4f} {final_recall:<15.4f} {final_recall - cv_performance['recall']:>+14.4f}")

## Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_holdout, y_holdout_pred)

print("\nConfusion Matrix:")
print(f"  TN: {cm[0,0]:5d}  |  FP: {cm[0,1]:5d}")
print(f"  FN: {cm[1,0]:5d}  |  TP: {cm[1,1]:5d}")

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Purchase', 'Purchase'],
            yticklabels=['No Purchase', 'Purchase'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix (Holdout Set)\nThreshold = {optimal_threshold:.2f}')
plt.tight_layout()
plt.savefig('../results/final_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nConfusion matrix saved to: results/final_confusion_matrix.png")

## Classification Report

In [ ]:
# Detailed classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_holdout, y_holdout_pred, 
                            target_names=['No Purchase', 'Purchase']))

## ROC Curve

In [ ]:
# Plot ROC curve
fpr, tpr, _ = roc_curve(y_holdout, y_holdout_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {final_roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Holdout Set)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/final_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print("ROC curve saved to: results/final_roc_curve.png")

## Error Analysis

In [ ]:
# Identify false positives and false negatives
fp_mask = (y_holdout == 0) & (y_holdout_pred == 1)
fn_mask = (y_holdout == 1) & (y_holdout_pred == 0)

print("\n" + "="*60)
print("ERROR ANALYSIS")
print("="*60)
print(f"\nFalse Positives: {fp_mask.sum()} ({fp_mask.sum()/len(y_holdout)*100:.2f}%)")
print(f"False Negatives: {fn_mask.sum()} ({fn_mask.sum()/len(y_holdout)*100:.2f}%)")

print(f"\nFalse Positive Rate: {cm[0,1]/(cm[0,0]+cm[0,1]):.4f}")
print(f"False Negative Rate: {cm[1,0]/(cm[1,0]+cm[1,1]):.4f}")

# Analyze prediction confidence for errors
if fp_mask.sum() > 0:
    fp_proba = y_holdout_proba[fp_mask]
    print(f"\nFalse Positive Confidence:")
    print(f"  Mean: {fp_proba.mean():.4f}")
    print(f"  Median: {np.median(fp_proba):.4f}")
    print(f"  Range: [{fp_proba.min():.4f}, {fp_proba.max():.4f}]")

if fn_mask.sum() > 0:
    fn_proba = y_holdout_proba[fn_mask]
    print(f"\nFalse Negative Confidence:")
    print(f"  Mean: {fn_proba.mean():.4f}")
    print(f"  Median: {np.median(fn_proba):.4f}")
    print(f"  Range: [{fn_proba.min():.4f}, {fn_proba.max():.4f}]")

## Save Final Results

In [ ]:
# Create final results dictionary
final_results = {
    'timestamp': datetime.now().isoformat(),
    'model': cv_performance['model_name'],
    'threshold': optimal_threshold,
    'holdout_metrics': {
        'f1_score': float(final_f1),
        'roc_auc': float(final_roc_auc),
        'precision': float(final_precision),
        'recall': float(final_recall),
        'accuracy': float(final_accuracy)
    },
    'cv_metrics': cv_performance,
    'comparison': {
        'f1_difference': float(final_f1 - cv_performance['f1_score']),
        'roc_auc_difference': float(final_roc_auc - cv_performance['roc_auc'])
    },
    'confusion_matrix': {
        'true_negatives': int(cm[0,0]),
        'false_positives': int(cm[0,1]),
        'false_negatives': int(cm[1,0]),
        'true_positives': int(cm[1,1])
    },
    'error_rates': {
        'false_positive_rate': float(cm[0,1]/(cm[0,0]+cm[0,1])),
        'false_negative_rate': float(cm[1,0]/(cm[1,0]+cm[1,1]))
    }
}

# Save to JSON
with open('../results/final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("\nFinal results saved to: results/final_results.json")

## Generate Final Report

In [ ]:
# Create markdown report
report = f"""
# Final Project Results

**Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Model Selection

**Best Model**: {cv_performance['model_name']}
**Optimal Threshold**: {optimal_threshold:.2f}

## Performance Summary

### Holdout Set Performance (Final)

| Metric | Value |
|--------|-------|
| F1-Score | {final_f1:.4f} |
| ROC-AUC | {final_roc_auc:.4f} |
| Precision | {final_precision:.4f} |
| Recall | {final_recall:.4f} |
| Accuracy | {final_accuracy:.4f} |

### Cross-Validation vs. Holdout

| Metric | CV (Stage 3) | Holdout | Difference |
|--------|--------------|---------|------------|
| F1-Score | {cv_performance['f1_score']:.4f} | {final_f1:.4f} | {final_f1 - cv_performance['f1_score']:+.4f} |
| ROC-AUC | {cv_performance['roc_auc']:.4f} | {final_roc_auc:.4f} | {final_roc_auc - cv_performance['roc_auc']:+.4f} |

## Confusion Matrix

```
                Predicted
              No      Yes
Actual  No  {cm[0,0]:5d}   {cm[0,1]:5d}
        Yes {cm[1,0]:5d}   {cm[1,1]:5d}
```

## Key Insights

1. **Model Generalization**: {'Good' if abs(final_f1 - cv_performance['f1_score']) < 0.02 else 'Moderate' if abs(final_f1 - cv_performance['f1_score']) < 0.05 else 'Poor'}
   - Difference between CV and Holdout F1: {abs(final_f1 - cv_performance['f1_score']):.4f}

2. **Error Analysis**:
   - False Positive Rate: {cm[0,1]/(cm[0,0]+cm[0,1]):.4f}
   - False Negative Rate: {cm[1,0]/(cm[1,0]+cm[1,1]):.4f}

3. **Business Impact** (for e-commerce stakeholder):
   - Of predicted purchases, {final_precision:.1%} are correct (Precision)
   - Of actual purchases, {final_recall:.1%} are caught (Recall)
   - Overall accuracy: {final_accuracy:.1%}

## Conclusion

The final model achieves {'excellent' if final_f1 > 0.60 else 'good' if final_f1 > 0.55 else 'acceptable'} performance with:
- F1-Score of {final_f1:.4f}
- ROC-AUC of {final_roc_auc:.4f}

{'The model generalizes well to unseen data.' if abs(final_f1 - cv_performance['f1_score']) < 0.02 else 'There is some performance degradation on holdout data, suggesting minor overfitting.'}
"""

with open('../results/final_report.md', 'w') as f:
    f.write(report)

print("Final report saved to: results/final_report.md")
print("\n" + report)

## 🎉 PROJECT COMPLETE!

**All Files Created**:
- ✅ `results/final_results.json` - Complete metrics
- ✅ `results/final_confusion_matrix.png` - Confusion matrix plot
- ✅ `results/final_roc_curve.png` - ROC curve
- ✅ `results/final_report.md` - Markdown report

**Next Steps**:
1. Review the final report
2. Analyze feature importances (if applicable)
3. Write up findings for stakeholder
4. Consider model deployment strategy